<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# HotelChain West Analytics
## Notebook 1 — SQL Analytics : KPIs, Performance & Analyses Avancées
### 📝 VERSION APPRENANT

> **Prérequis** : avoir lu le contexte métier et le dictionnaire des données sur la plateforme.
>
> **Instructions :** Complete les cellules marquées `# TODO`. Les blocs `MÉTHODE` t'expliquent l'approche attendue.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), pandas |
| **Durée estimée** | 4h à 5h |

---
## 0. Mise en place de l'environnement

In [1]:
!pip install jupysql==0.11.1 duckdb-engine --quiet


[notice] A new release of pip is available: 24.1 -> 26.0.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import os, sys, warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
}

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/hotelchain_analytics/"
else:
    SAVE_PATH = "./outputs/"
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"📁 Environnement : {'Colab' if IN_COLAB else 'Local'}")
print(f"📁 Dossier       : {SAVE_PATH}")
print("Environnement prêt ✅")

📁 Environnement : Local
📁 Dossier       : ./outputs/
Environnement prêt ✅


---
## 1. Connexion DuckDB et chargement des tables

> **MÉTHODE — DuckDB remplace SQL Server dans ce notebook.**
>
> DuckDB est une base de données SQL embarquée optimisée pour l'analytique. Elle charge les CSV directement depuis GitHub via `read_csv_auto()` — zéro installation, compatible Colab et local.
>
> **Adaptations syntaxiques vs SQL Server :**
>
> | SQL Server | DuckDB |
> |---|---|
> | `CREATE OR ALTER VIEW` | `CREATE OR REPLACE VIEW` |
> | `TOP N` | `LIMIT N` |
> | `EOMONTH()` | `LAST_DAY()` |
> | `PIVOT` | pandas `.pivot_table()` |
> | `CREATE PROCEDURE` | fonction Python |
> | `BULK INSERT` | `read_csv_auto()` |

In [3]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/data/"

# Créer la connexion et charger les 6 tables
conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE hotels       AS SELECT * FROM read_csv_auto('{BASE_URL}hotels.csv');
    CREATE TABLE chambres     AS SELECT * FROM read_csv_auto('{BASE_URL}chambres.csv');
    CREATE TABLE clients      AS SELECT * FROM read_csv_auto('{BASE_URL}clients.csv');
    CREATE TABLE reservations AS SELECT * FROM read_csv_auto('{BASE_URL}reservations.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE paiements    AS SELECT * FROM read_csv_auto('{BASE_URL}paiements.csv');
    CREATE TABLE services     AS SELECT * FROM read_csv_auto('{BASE_URL}services.csv');
""")

# Vérification
result = conn.execute("""
    SELECT 'hotels'       AS table_name, COUNT(*) AS nb_lignes FROM hotels       UNION ALL
    SELECT 'chambres',      COUNT(*) FROM chambres     UNION ALL
    SELECT 'clients',       COUNT(*) FROM clients      UNION ALL
    SELECT 'reservations',  COUNT(*) FROM reservations UNION ALL
    SELECT 'paiements',     COUNT(*) FROM paiements    UNION ALL
    SELECT 'services',      COUNT(*) FROM services
""").df()
display(result)
print("✅ 6 tables chargées")

,table_name,nb_lignes
0,hotels,5
1,chambres,515
2,clients,3000
3,reservations,8000
4,paiements,9698
5,services,9001


✅ 6 tables chargées


### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

In [4]:
# Lier DuckDB à JupySQL pour les cellules %%sql
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print("%%sql prêt ✅")

---
## 2. Vues de nettoyage

> **MÉTHODE — Pourquoi des vues plutôt que modifier les données brutes ?**
>
> Les vues sont des requêtes sauvegardées qui se comportent comme des tables. Les données originales restent intactes et reproductibles. Toute l'analyse s'appuie sur ces vues nettoyées.

### Vue 1 — Clients propres (sans doublons email, âges valides)

In [5]:
%%sql
CREATE OR REPLACE VIEW vw_clients_propres AS
SELECT *
FROM clients
WHERE age BETWEEN 16 AND 80
  AND client_id NOT IN (
      -- TODO : sous-requête pour exclure les doublons email
      -- Indice : ROW_NUMBER() OVER (PARTITION BY email ORDER BY client_id)
      -- Garder le plus petit client_id pour chaque email
  );

### Vue 2 — Réservations propres (montants > 0, nb_nuits > 0)

In [6]:
%%sql
CREATE OR REPLACE VIEW vw_reservations_propres AS
SELECT *
FROM reservations
WHERE -- TODO : exclure montants négatifs
  AND -- TODO : exclure nb_nuits = 0

### Vue 3 — Paiements valides

In [7]:
%%sql
CREATE OR REPLACE VIEW vw_paiements_valides AS
SELECT *
FROM paiements
WHERE -- TODO : garder uniquement statut_paiement = 'Valide'

In [8]:
# Vérification avant/après nettoyage
check = conn.execute("""
    SELECT 'clients brut'        AS source, COUNT(*) AS nb FROM clients         UNION ALL
    SELECT 'clients propres',      COUNT(*) FROM vw_clients_propres             UNION ALL
    SELECT 'reservations brut',    COUNT(*) FROM reservations                   UNION ALL
    SELECT 'reservations propres', COUNT(*) FROM vw_reservations_propres        UNION ALL
    SELECT 'paiements brut',       COUNT(*) FROM paiements                      UNION ALL
    SELECT 'paiements valides',    COUNT(*) FROM vw_paiements_valides
""").df()
display(check)

,source,nb
0,clients brut,3000
1,clients propres,2993
2,reservations brut,8000
3,reservations propres,7992
4,paiements brut,9698
5,paiements valides,9348


### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 3. KPIs globaux en une seule requête

> **MÉTHODE — Une seule requête pour tous les KPIs de surface.**
>
> C'est la première réponse au Directeur Général : *'Quel est l'état de mon business ?'*  
> On filtre sur les réservations avec statut `'Terminee'` pour les KPIs financiers.

In [9]:
%%sql df_kpi <<
SELECT
    COUNT(DISTINCT r.reservation_id)   AS total_reservations,
    -- TODO : revenu_total (SUM des paiements valides)
    -- TODO : csat_moyen (AVG sur réservations terminées)
    -- TODO : duree_moyenne_sejour
    -- TODO : nb_clients_uniques
    -- TODO : revenu_moyen_par_reservation
    -- TODO : taux_annulation_pct (% statut = 'Annulee')
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 4. Taux d'occupation par hôtel et par mois

> **MÉTHODE — Taux d'occupation = nuits vendues / (nb_chambres × nb_jours_du_mois) × 100**
>
> On utilise `RANK() OVER (PARTITION BY annee, mois ORDER BY taux DESC)` pour classer les hôtels chaque mois — sans perdre les lignes des autres hôtels, contrairement à `LIMIT`.
>
> **DuckDB :** `LAST_DAY(DATE_TRUNC('month', date_arrivee))` remplace `EOMONTH()` de SQL Server.

In [10]:
%%sql df_occupation <<
WITH nuits_vendues AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)  AS annee,
        MONTH(r.date_arrivee) AS mois,
        -- TODO : SUM des nuits vendues
    FROM vw_reservations_propres r
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
),
capacite AS (
    -- TODO : calculer capacite_theorique = nb_chambres × nb_jours_du_mois
    -- Indice DuckDB : DAY(LAST_DAY(MAKE_DATE(annee, mois, 1)))
    SELECT nv.hotel_id, nv.annee, nv.mois,
           h.nb_chambres,
           -- TODO : nb_jours_mois
           -- TODO : capacite_theorique
    FROM nuits_vendues nv
    JOIN hotels h ON nv.hotel_id = h.hotel_id
)
SELECT
    h.nom,
    nv.annee,
    nv.mois,
    nv.nuits_vendues,
    c.capacite_theorique,
    -- TODO : taux_occupation_pct = nuits_vendues / capacite_theorique × 100
    -- TODO : RANK() OVER (PARTITION BY annee, mois ORDER BY taux DESC) AS rang_mois
FROM nuits_vendues nv
JOIN hotels h    ON nv.hotel_id = h.hotel_id
JOIN capacite c  ON nv.hotel_id = c.hotel_id AND nv.annee = c.annee AND nv.mois = c.mois
ORDER BY nv.annee, nv.mois, taux_occupation_pct DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

In [11]:
# Visualisation : taux d'occupation mensuel par hôtel
df_occ_pivot = df_occupation.pivot_table(
    index=['annee', 'mois'], columns='nom', values='taux_occupation_pct'
).reset_index()
df_occ_pivot['periode'] = df_occ_pivot['annee'].astype(str) + '-' + df_occ_pivot['mois'].astype(str).str.zfill(2)

hotels_cols = [c for c in df_occ_pivot.columns if c not in ['annee', 'mois', 'periode']]
fig, ax = plt.subplots(figsize=(14, 5))
for col in hotels_cols:
    ax.plot(df_occ_pivot['periode'], df_occ_pivot[col], marker='o', markersize=3, linewidth=1.5, label=col)
ax.axhline(70, color=COLORS['secondary'], linestyle='--', linewidth=1.5, label='Cible 70%')
ax.axhline(40, color=COLORS['danger'], linestyle='--', linewidth=1.5, label='Seuil alerte 40%')
ax.set_xticks(range(0, len(df_occ_pivot), 3))
ax.set_xticklabels(df_occ_pivot['periode'].iloc[::3], rotation=45, ha='right', fontsize=9)
ax.set_title('Taux d\'occupation mensuel par hôtel', fontweight='bold', fontsize=13)
ax.set_ylabel('Taux d\'occupation (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}taux_occupation.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Performance par canal — RANK() et LAG()

> **MÉTHODE — Deux questions distinctes sur les canaux :**
>
> **RANK() :** quel canal est le plus rentable globalement ?  
> **LAG() :** comment évolue chaque canal mois par mois ?

In [12]:
%%sql df_canal <<
SELECT
    r.canal,
    COUNT(*)             AS nb_reservations,
    -- TODO : revenu_total
    -- TODO : revenu_moyen
    -- TODO : taux_annulation_pct
    -- TODO : RANK() OVER (ORDER BY revenu_total DESC) AS rang_revenu
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
GROUP BY r.canal
ORDER BY revenu_total DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

In [13]:
%%sql df_canal_lag <<
-- Tendance mensuelle par canal avec LAG()
WITH mensuel AS (
    SELECT
        r.canal,
        YEAR(r.date_arrivee)  AS annee,
        MONTH(r.date_arrivee) AS mois,
        -- TODO : revenu_mensuel
    FROM vw_reservations_propres r
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    GROUP BY r.canal, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
)
SELECT
    canal, annee, mois, revenu_mensuel,
    -- TODO : LAG(revenu_mensuel) OVER (PARTITION BY canal ORDER BY annee, mois)
    -- TODO : variation = revenu_mensuel - revenu_mois_prec
FROM mensuel
ORDER BY canal, annee, mois

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 6. RevPAR — Revenue Per Available Room

> **MÉTHODE — RevPAR = ADR × Taux d'occupation**
>
> C'est l'indicateur standard de l'industrie hôtelière. Il combine à la fois la politique tarifaire (ADR) et le remplissage (taux d'occupation) en une seule métrique.
>
> - **ADR élevé + taux bas** : hotel trop cher, peu rempli
> - **ADR bas + taux haut** : hôtel sous-tarifé, bien rempli mais peu rentable
> - **ADR élevé + taux élevé** : sweet spot tarifaire

In [14]:
%%sql df_revpar <<
WITH adr_calc AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)  AS annee,
        MONTH(r.date_arrivee) AS mois,
        -- TODO : adr = SUM(montant) / SUM(nb_nuits)
        -- TODO : nuits_vendues = SUM(nb_nuits)
    FROM vw_reservations_propres r
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
),
capacite AS (
    -- TODO : même calcul de capacité_theorique qu'en section 4
    SELECT a.hotel_id, a.annee, a.mois,
           -- TODO : capacite_theorique
    FROM adr_calc a
    JOIN hotels h ON a.hotel_id = h.hotel_id
)
SELECT
    h.nom, a.annee, a.mois, a.adr,
    -- TODO : taux_occupation_pct
    -- TODO : revpar = adr × (nuits_vendues / capacite_theorique)
    -- TODO : RANK() OVER (PARTITION BY annee, mois ORDER BY revpar DESC)
FROM adr_calc a
JOIN hotels h    ON a.hotel_id = h.hotel_id
JOIN capacite c  ON a.hotel_id = c.hotel_id AND a.annee = c.annee AND a.mois = c.mois
ORDER BY a.annee, a.mois, revpar DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 7. Segmentation clients — NTILE et valeur client (CLV)

> **MÉTHODE — `NTILE(4) OVER (ORDER BY revenu_total DESC)` : segmentation en quartiles.**
>
> Quartile 1 = top 25% des clients en valeur → segment à protéger.  
> `PERCENT_RANK()` pour identifier le top 10% (VIP).

In [15]:
%%sql df_clv <<
WITH clv AS (
    SELECT
        c.client_id,
        c.prenom || ' ' || c.nom  AS client_nom,
        c.nationalite,
        c.client_fidele,
        -- TODO : nb_sejours
        -- TODO : revenu_total
        -- TODO : duree_moyenne
        -- TODO : csat_moyen
    FROM vw_clients_propres c
    JOIN vw_reservations_propres r   ON c.client_id = r.client_id
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY c.client_id, c.prenom, c.nom, c.nationalite, c.client_fidele
)
SELECT *,
    -- TODO : NTILE(4) OVER (ORDER BY revenu_total DESC) AS quartile_valeur
    -- TODO : CASE WHEN PERCENT_RANK() <= 0.10 THEN 'VIP' ELSE 'Standard' END
FROM clv
ORDER BY revenu_total DESC
LIMIT 20

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 8. Heatmap saisonnalité — mois × hôtel

> **MÉTHODE — Le PIVOT SQL Server n'existe pas dans DuckDB.**
>
> On utilise `pandas.pivot_table()` — résultat identique, code plus lisible.

In [16]:
# Récupérer les données pour la heatmap
df_heatmap_raw = conn.execute("""
    SELECT
        h.nom                                                          AS hotel,
        MONTH(r.date_arrivee)                                          AS mois,
        ROUND(
            SUM(r.nb_nuits) * 100.0 /
            (h.nb_chambres * DAY(LAST_DAY(MAKE_DATE(YEAR(r.date_arrivee), MONTH(r.date_arrivee), 1)))),
        1) AS taux_occupation
    FROM vw_reservations_propres r
    JOIN hotels h ON r.hotel_id = h.hotel_id
    WHERE r.statut IN ('Terminee', 'En cours')
      AND YEAR(r.date_arrivee) = 2023
    GROUP BY h.nom, h.nb_chambres, MONTH(r.date_arrivee), YEAR(r.date_arrivee)
""").df()

In [17]:
import seaborn as sns

# Pivot pandas
pivot = df_heatmap_raw.pivot_table(index='hotel', columns='mois', values='taux_occupation')
pivot.columns = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=30, vmax=90, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Taux occupation (%)'}
)
ax.set_title('Heatmap saisonnalité — Taux d\'occupation 2023 (mois × hôtel)', fontweight='bold', fontsize=13)
ax.set_xlabel('Mois')
ax.set_ylabel('Hôtel')
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}heatmap_saisonnalite.png", dpi=150, bbox_inches='tight')
plt.show()

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 9. Rapport hôtel — fonction Python (remplace la procédure stockée)

> **MÉTHODE — `CREATE PROCEDURE` n'existe pas dans DuckDB.**
>
> On crée une fonction Python qui accepte un `hotel_id` et retourne le rapport complet — exactement le même résultat qu'une procédure stockée SQL Server, avec plus de flexibilité.

In [18]:
def rapport_hotel(hotel_id: str, annee: int = 2023):
    """Génère le rapport complet pour un hôtel.
    Équivalent de sp_rapport_hotel en SQL Server.
    """
    return conn.execute(f"""
        -- TODO : requête qui retourne pour un hôtel et une année donnés :
        -- nom, mois, nb_reservations, nuits_vendues, capacite,
        -- revenu, csat_moyen, adr, taux_occupation_pct, revpar
        -- Indice : reprendre la logique des sections 4 et 6
        SELECT 'TODO' AS nom
        WHERE hotel_id = '{hotel_id}' AND YEAR(date_arrivee) = {annee}
    """).df()

# Test sur le premier hôtel
df_rapport = rapport_hotel('HTL001', 2023)
display(df_rapport)

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la requête)*

- 
- 
- 

---
## 10. Export des datasets analytiques

> Ces fichiers seront importés dans Power BI au Notebook 2.

In [19]:
# Export des datasets analytiques pour Power BI
# À exécuter une fois les cellules précédentes complétées

df_occupation.to_csv(f"{SAVE_PATH}hotelchain_occupation.csv", index=False)
df_revpar.to_csv(f"{SAVE_PATH}hotelchain_revpar.csv", index=False)
df_clv.to_csv(f"{SAVE_PATH}hotelchain_clients_clv.csv", index=False)
df_canal.to_csv(f"{SAVE_PATH}hotelchain_canaux.csv", index=False)

print("Fichiers exportés ✅")
print(f"📁 Localisation : {SAVE_PATH}")

---
## Checklist de validation

| Étape | Contenu | Statut |
|---|---|---|
| 2 | 3 vues de nettoyage créées | [ ] |
| 3 | KPIs globaux calculés | [ ] |
| 4 | Taux d'occupation mensuel + visualisation | [ ] |
| 5 | Performance canaux avec RANK() et LAG() | [ ] |
| 6 | RevPAR calculé par hôtel et par mois | [ ] |
| 7 | Segmentation clients NTILE + VIP | [ ] |
| 8 | Heatmap saisonnalité produite | [ ] |
| 9 | Fonction rapport_hotel() testée | [ ] |
| 10 | 4 fichiers CSV exportés | [ ] |

| KPI | Valeur |
|---|---|
| Revenu total | ? FCFA |
| CSAT moyen | ? / 5 |
| Taux annulation | ? % |
| Hôtel meilleur RevPAR | ? |
| Canal le plus rentable | ? |

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.